In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

In [ ]:
# 1. 설정값 정의
model_id = "yanolja/YanoljaNEXT-EEVE-Instruct-10.8B"
dataset_path = "text_training_data.jsonl" 

In [ ]:
# 2. 모델 로드 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# 3. 데이터 포맷팅 함수 
def formatting_prompts_func(example):
    output_texts = []
    for i in range(len(example['instruction'])):
        text = f"### 지시:\n{example['instruction'][i]}\n\n### 입력:\n{example['input'][i]}\n\n### 답변:\n{example['output'][i]}<|end_of_text|>"
        output_texts.append(text)
    return output_texts

In [ ]:
# 4. LoRA 설정 
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
# 5. 학습 인자 설정
training_args = TrainingArguments(
    output_dir="./eeve-persona-results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4, 
    logging_steps=10,
    num_train_epochs=3,
    save_strategy="steps",
    save_steps=100,
    lr_scheduler_type="cosine",
    optim="paged_adamw_32bit",
    fp16=True, 
)

In [ ]:
# 6. 트레이너 실행
dataset = load_dataset("json", data_files=dataset_path, split="train")

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=formatting_prompts_func,
    args=training_args,
)

trainer.train()

In [ ]:
# 7. 학습된 어댑터 저장
trainer.model.save_pretrained("./final_persona_model")

### 허깅페이스에 모델 업로드

In [1]:
# 허깅페이스 업로드
from huggingface_hub import HfApi, login
import os
from dotenv import load_dotenv

load_dotenv()

# 설정값 
HF_USERNAME = "HyojungJ"  # 허깅페이스 사용자명
MODEL_NAME = "eeve-persona-memefluencer"  # 모델 이름
HF_TOKEN = os.getenv("HF_TOKEN")

# 업로드할 폴더와 저장소 ID
model_folder = "../../data/finetuning/final_persona_model"
repo_id = f"{HF_USERNAME}/{MODEL_NAME}"

# 허깅페이스 로그인
login(token=HF_TOKEN)
api = HfApi()

# 저장소 생성 및 업로드
api.create_repo(repo_id=repo_id, exist_ok=True)
api.upload_folder(
    folder_path=model_folder,
    repo_id=repo_id,
    repo_type="model"
)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


adapter_model.safetensors:   0%|          | 0.00/126M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/HyojungJ/eeve-persona-memefluencer/commit/1c4f660dac9d0d903b962544a7cbaf276fcce9a7', commit_message='Upload folder using huggingface_hub', commit_description='', oid='1c4f660dac9d0d903b962544a7cbaf276fcce9a7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/HyojungJ/eeve-persona-memefluencer', endpoint='https://huggingface.co', repo_type='model', repo_id='HyojungJ/eeve-persona-memefluencer'), pr_revision=None, pr_num=None)

### 업로드된 모델 사용 방법

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# 베이스 모델 로드
base_model_id = "yanolja/YanoljaNEXT-EEVE-Instruct-10.8B"
model = AutoModelForCausalLM.from_pretrained(base_model_id)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# 파인튜닝된 어댑터 로드
model = PeftModel.from_pretrained(model, "your_username/eeve-persona-lora")

# 추론
prompt = "### 지시:\n너는 광고회사 부장이야\n\n### 입력:\n신입사원이 피곤하다고 한다\n\n### 답변:\n"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_length=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))